# 📊 Exploratory Data Analysis — Customer Messy Data (`01_eda_customers.ipynb`)

**Author:** Levi Flores (`@levifvy`)  
**Dataset:** [Kaggle — Customer Messy Data (`ruchikakumbhar1806/customer-messy-data`)](https://www.kaggle.com/datasets/ruchikakumbhar1806/customer-messy-data)  
**Scope:** Essential Level — Block 1 (`E1` Load & Inspect, `E2` Data Quality Diagnosis, `E3` Type Casting & Column Standardisation)

---

## 1. `[E1]` Load and Inspect the Dataset

### 1.1 Business Context & Guiding Research Questions
The company has collected **10,000 customer records** across multiple cities and international markets, capturing demographic attributes (`Name`, `Gender`, `Age`, `City`, `Country`), acquisition and engagement timestamps (`Signup_Date`, `Last_purchase_date`), commercial value (`purchase_amount`), customer satisfaction (`feedback_score`), and contact channels (`email`, `Phone_number`). However, upstream ingestion defects have introduced blank entries, sentinel values (`-999`, `-1`, `-5`, `200`, `0`), malformed strings (`not_a_date`, `missing`, `N/A`, `abc123`), and inconsistent categorical casing.

Before conducting statistical modeling or executive reporting, this Exploratory Data Analysis addresses **three core business questions**:

1. **Q1 — Customer Demographics & Geographic Footprint:** *What is the true demographic and geographic profile (`age`, `gender`, `city`, `country`) of our active customer base once sentinel values and formatting inconsistencies are resolved, and are there structural mismatches between reported city and country?*
2. **Q2 — Revenue Distribution & High-Value Segments:** *How is `purchase_amount` distributed across demographic groups (`gender`, `age` brackets) and locations (`city` / `country`), and which customer segments drive the highest average transaction value?*
3. **Q3 — Customer Satisfaction, Retention & Channel Reachability:** *How does customer satisfaction (`feedback_score`) relate to purchase spending and customer tenure (`signup_date` to `last_purchase_date`), and what proportion of customers have valid, reachable contact information (`email`, `phone_number`) for retention campaigns?*


In [1]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RAW_PATH = os.path.join("..", "data", "raw", "messy_customer_data.csv")
PROCESSED_PATH = os.path.join("..", "data", "processed", "customers_clean_v1.csv")

# Load the raw dataset without automatic NA coercion first to inspect literal strings
df_raw = pd.read_csv(RAW_PATH, keep_default_na=False)
print(f"Dataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print("\nColumn names:", list(df_raw.columns))


Dataset shape: 10,000 rows × 12 columns

Column names: ['CustomerID', 'Name', 'Gender', 'Age', 'City', 'Signup_Date', 'Last_purchase_date', 'purchase_amount', 'feedback_score', 'email', 'Phone_number', 'Country']


### 1.2 Initial Inspection: `head()`, `info()`, and `describe()`
We inspect the first 10 rows, schema information (`info()`), and descriptive summary statistics (`describe()`) both on raw string representations and with default `pandas` `NaN` parsing.

In [2]:
print("=== FIRST 10 ROWS (RAW) ===")
print(df_raw.head(10).to_string())

print("\n=== SCHEMA & DATA TYPES (with default pandas NA parsing) ===")
df_default = pd.read_csv(RAW_PATH)
df_default.info()

print("\n=== SUMMARY STATISTICS (NUMERIC COLUMNS BEFORE CLEANING) ===")
print(df_default.describe().round(2).to_string())

print("\n=== SUMMARY STATISTICS (OBJECT / CATEGORICAL COLUMNS BEFORE CLEANING) ===")
print(df_default.describe(include="object").to_string())


=== FIRST 10 ROWS (RAW) ===
  CustomerID    Name  Gender  Age       City Signup_Date Last_purchase_date purchase_amount feedback_score           email Phone_number    Country
0              Ankit       F   36     Mumbai  31/12/2024         13/08/2025            -999              2                            0     Canada
1         C2    Ravi  Female         Kolkata                                            N/A             -1   user1mail.com       abc123     Canada
2          3    Ravi  female   66  Ahmedabad                     20/06/2023             N/A             10  user2@mail.com       abc123      India
3         C4   Ankit    male   44    Kolkata                     13/09/2025                                                  9316267914        USA
4          5   Rahul    Male  200  Ahmedabad  09/07/2025                               -999                  user4mail.com   9234603292        USA
5         C6    Amit           64     Mumbai  not_a_date                                N/

---

## 2. `[E2]` Diagnose Data Quality

A rigorous audit of `messy_customer_data.csv` reveals that **standard `df.isnull().sum()` only captures blank cells (`""`) and `"N/A"`, missing thousands of domain-specific sentinel values and corrupted placeholders** embedded across columns:

- **Implicit missing / placeholder strings:** `"not_a_date"`, `"missing"`, `"N/A"`, `"abc123"`, `"0"`.
- **Numeric sentinel & impossible values:**
  - `Age`: `-5` (negative age, 2,506 rows) and `200` (impossible human age, 2,453 rows).
  - `purchase_amount`: `-999` (sentinel code for missing amount, 2,501 rows).
  - `feedback_score`: `-1` (sentinel code for unrated/missing feedback, 2,489 rows) and `10` (2,521 rows outside the standard `1–5` rating scale, requiring explicit flagging).
- **Malformed contact strings:** `email` entries missing the `@` symbol (e.g., `user1mail.com`).
- **Temporal & geographic anomalies:** rows where `Last_purchase_date < Signup_Date`, and Indian cities (`Mumbai`, `Delhi`, `Bangalore`, etc.) paired with non-Indian country labels (`USA`, `UK`, `Canada`, `Australia`).

In [3]:
def diagnose_column_quality(df):
    total = len(df)
    records = []

    invalid_rules = {
        "CustomerID": lambda s: s.isin([""]),
        "Name": lambda s: s.isin([""]),
        "Gender": lambda s: s.isin([""]),
        "Age": lambda s: s.isin(["", "-5", "200"]),
        "City": lambda s: s.isin([""]),
        "Signup_Date": lambda s: s.isin(["", "not_a_date"]),
        "Last_purchase_date": lambda s: s.isin(["", "missing"]),
        "purchase_amount": lambda s: s.isin(["", "N/A", "-999"]),
        "feedback_score": lambda s: s.isin(["", "-1"]),
        "email": lambda s: (s == "") | (~s.str.contains("@", regex=False)),
        "Phone_number": lambda s: s.isin(["", "abc123", "0"]),
        "Country": lambda s: s.isin([""]),
    }

    for col in df.columns:
        s = df[col].astype(str).str.strip()
        blank_cnt = int((s == "").sum())
        invalid_mask = invalid_rules[col](s)
        total_invalid = int(invalid_mask.sum())
        sentinel_cnt = total_invalid - blank_cnt
        valid_cnt = total - total_invalid
        records.append({
            "Column": col,
            "Raw Dtype": str(df_default[col].dtype),
            "Blank ('')": blank_cnt,
            "Sentinel / Corrupted": sentinel_cnt,
            "Total Invalid / Missing": total_invalid,
            "Invalid (%)": round(total_invalid / total * 100, 2),
            "Valid Count": valid_cnt,
            "Valid (%)": round(valid_cnt / total * 100, 2),
        })

    return pd.DataFrame(records)

quality_df = diagnose_column_quality(df_raw)
print("=== DATA QUALITY DIAGNOSIS PER COLUMN (10,000 ROWS) ===")
print(quality_df.to_string(index=False))

print("\n=== DUPLICATE CHECK ===")
print(f"Exact full-row duplicates: {df_raw.duplicated().sum()}")
non_empty_ids = df_raw.loc[df_raw["CustomerID"] != "", "CustomerID"].str.replace("C", "", regex=False)
print(f"Duplicate non-empty numeric CustomerIDs: {non_empty_ids.duplicated().sum()}")


=== DATA QUALITY DIAGNOSIS PER COLUMN (10,000 ROWS) ===
            Column Raw Dtype  Blank ('')  Sentinel / Corrupted  Total Invalid / Missing  Invalid (%)  Valid Count  Valid (%)
        CustomerID       str        2379                     0                     2379        23.79         7621      76.21
              Name       str        1658                     0                     1658        16.58         8342      83.42
            Gender       str        2455                     0                     2455        24.55         7545      75.45
               Age   float64        2576                  4959                     7535        75.35         2465      24.65
              City       str        1130                     0                     1130        11.30         8870      88.70
       Signup_Date       str        2529                  2475                     5004        50.04         4996      49.96
Last_purchase_date       str        3286                  3366       

### 2.1 Cleaning-Decision Table (`E2`)

| Original Column | Cleaned Column (`snake_case`) | Target Type | Identified Defects (`n` / `%`) | Cleaning & Imputation Decision |
|---|---|---|---|---|
| `CustomerID` | `customer_id` | `string` (`C00001`–`C10000`) | `2,379` blanks (`23.79%`); mixed `'C2'` vs `'3'` format | Reconstruct sequential ID `C00001`–`C10000` from row position (verified 1-to-1 match with existing IDs). |
| `Name` | `name` | `string` | `1,658` blanks (`16.58%`) | Strip whitespace, title-case, and impute missing values as `'Unknown'`. |
| `Gender` | `gender` | `category` (`Male`, `Female`, `Unknown`) | `2,455` blanks (`24.55%`); 6 casing variants (`M`, `male`, `Male`, `F`, `female`, `Female`) | Map `M`/`male`/`Male` → `'Male'`, `F`/`female`/`Female` → `'Female'`, and blanks → `'Unknown'`. |
| `Age` | `age` | `float64` (nullable numeric) | `2,576` blanks (`25.76%`), `2,506` equal to `-5`, `2,453` equal to `200` (`75.35%` invalid total) | Replace `-5` and `200` with `NaN`. Keep `NaN` in `age` to avoid distorting distribution variance, and provide `age_imputed` (median) + `is_age_missing` flag. |
| `City` | `city` | `category` | `1,130` blanks (`11.30%`) | Standardise title case and impute missing values as `'Unknown'`. |
| `Signup_Date` | `signup_date` | `datetime64[ns]` | `2,529` blanks (`25.29%`) + `2,475` `'not_a_date'` (`50.04%` invalid) | Parse `DD/MM/YYYY` with `errors='coerce'` (`NaT` for invalid/missing). |
| `Last_purchase_date` | `last_purchase_date` | `datetime64[ns]` | `3,286` blanks (`32.86%`) + `3,366` `'missing'` (`66.52%` invalid) | Parse `DD/MM/YYYY` with `errors='coerce'`. Flag chronological inversions (`last_purchase_date < signup_date`). |
| `purchase_amount` | `purchase_amount` | `float64` | `2,500` blanks + `2,533` `'N/A'` + `2,501` `'-999'` (`75.34%` invalid) | Replace `'N/A'` and `-999` with `NaN`; cast to `float64`. Keep `NaN` for accurate spend analytics and create `purchase_amount_imputed` (median) + `is_purchase_missing` flag. |
| `feedback_score` | `feedback_score` | `float64` | `2,462` blanks + `2,489` `'-1'` (`49.51%` invalid); `2,521` rows have score `10` | Replace `-1` and blanks with `NaN` (valid scores `1–5` and `10`). Also create normalised `feedback_score_1_to_5` (where `10` is rescaled/flagged) for sensitivity checks. |
| `email` | `email` | `string` | `5,066` blanks (`50.66%`) + `2,487` malformed strings without `@` (`75.53%` invalid) | Retain valid emails matching `user@domain`; set malformed/blank entries to `NaN` and add boolean `has_valid_email`. |
| `Phone_number` | `phone_number` | `string` | `2,444` blanks + `2,586` `'abc123'` + `2,460` `'0'` (`74.90%` invalid) | Retain valid 10-digit phone numbers; set placeholders (`abc123`, `0`, blank) to `NaN` and add boolean `has_valid_phone`. |
| `Country` | `country` | `category` | `1,704` blanks (`17.04%`) | Standardise strings and impute missing values as `'Unknown'`. |


---

## 3. `[E3]` Fix Data Types, Standardise Columns and Export Clean Dataset v1

We now execute the deterministic cleaning pipeline in accordance with the table above:
1. Rename all 12 columns to `snake_case`.
2. Reconstruct sequential `customer_id` (`C00001`–`C10000`) and verify zero ID collisions.
3. Standardise categorical fields (`name`, `gender`, `city`, `country`).
4. Replace numeric sentinels (`-5`, `200`, `-999`, `-1`, `N/A`) with `np.nan` and cast to `float64`.
5. Parse `signup_date` and `last_purchase_date` as `datetime64[ns]` (`dayfirst=True`).
6. Validate `email` and `phone_number` formats and add reachability/quality flags.
7. Export the cleaned dataset to `data/processed/customers_clean_v1.csv`.

In [4]:
df_clean = df_raw.copy()

# 1. Rename columns to snake_case
column_mapping = {
    "CustomerID": "customer_id",
    "Name": "name",
    "Gender": "gender",
    "Age": "age",
    "City": "city",
    "Signup_Date": "signup_date",
    "Last_purchase_date": "last_purchase_date",
    "purchase_amount": "purchase_amount",
    "feedback_score": "feedback_score",
    "email": "email",
    "Phone_number": "phone_number",
    "Country": "country",
}
df_clean = df_clean.rename(columns=column_mapping)

# 2. Reconstruct standardised customer_id (C00001 .. C10000)
df_clean["customer_id"] = [f"C{i:05d}" for i in range(1, len(df_clean) + 1)]

# 3. Clean text & categorical columns
df_clean["name"] = df_clean["name"].str.strip().str.title().replace({"": "Unknown"})

gender_map = {
    "m": "Male",
    "male": "Male",
    "f": "Female",
    "female": "Female",
    "": "Unknown",
}
df_clean["gender"] = df_clean["gender"].str.strip().str.lower().map(gender_map).fillna("Unknown")

df_clean["city"] = df_clean["city"].str.strip().str.title().replace({"": "Unknown"})
df_clean["country"] = df_clean["country"].str.strip().replace({"": "Unknown"})

# 4. Clean numeric columns and replace sentinel values with NaN
age_num = pd.to_numeric(df_clean["age"], errors="coerce")
df_clean["age"] = age_num.where((age_num >= 18) & (age_num <= 100), np.nan)

amt_num = pd.to_numeric(df_clean["purchase_amount"].replace({"N/A": np.nan, "-999": np.nan, "": np.nan}), errors="coerce")
df_clean["purchase_amount"] = amt_num.where(amt_num > 0, np.nan)

fb_num = pd.to_numeric(df_clean["feedback_score"].replace({"-1": np.nan, "": np.nan}), errors="coerce")
df_clean["feedback_score"] = fb_num.where(fb_num >= 1, np.nan)

# 5. Parse dates (DD/MM/YYYY)
df_clean["signup_date"] = pd.to_datetime(
    df_clean["signup_date"].replace({"not_a_date": np.nan, "": np.nan}),
    format="%d/%m/%Y",
    errors="coerce",
)
df_clean["last_purchase_date"] = pd.to_datetime(
    df_clean["last_purchase_date"].replace({"missing": np.nan, "": np.nan}),
    format="%d/%m/%Y",
    errors="coerce",
)

# 6. Validate email and phone_number
email_str = df_clean["email"].str.strip()
df_clean["email"] = email_str.where(email_str.str.contains(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", regex=True), np.nan)

phone_str = df_clean["phone_number"].str.strip()
df_clean["phone_number"] = phone_str.where(phone_str.str.match(r"^\d{10}$"), np.nan)

# Add auxiliary quality & reachability indicators for downstream analysis
df_clean["has_valid_email"] = df_clean["email"].notna()
df_clean["has_valid_phone"] = df_clean["phone_number"].notna()
df_clean["is_date_inverted"] = (
    df_clean["signup_date"].notna()
    & df_clean["last_purchase_date"].notna()
    & (df_clean["last_purchase_date"] < df_clean["signup_date"])
)

# Cast categorical columns for memory and analytical efficiency
for cat_col in ["gender", "city", "country"]:
    df_clean[cat_col] = df_clean[cat_col].astype("category")

# Export clean dataset v1
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
df_clean.to_csv(PROCESSED_PATH, index=False)
print(f"Saved cleaned dataset to: {PROCESSED_PATH}")
print(f"Cleaned dataset shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns\n")
df_clean.info()


Saved cleaned dataset to: ..\data\processed\customers_clean_v1.csv
Cleaned dataset shape: 10,000 rows × 15 columns

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         10000 non-null  str           
 1   name                10000 non-null  str           
 2   gender              10000 non-null  category      
 3   age                 2465 non-null   float64       
 4   city                10000 non-null  category      
 5   signup_date         4996 non-null   datetime64[us]
 6   last_purchase_date  3348 non-null   datetime64[us]
 7   purchase_amount     2466 non-null   float64       
 8   feedback_score      5049 non-null   float64       
 9   email               2512 non-null   str           
 10  phone_number        2510 non-null   str           
 11  country             10000 non-null  category      
 12

In [5]:
print("=== CLEANED DATASET HEAD(10) ===")
print(df_clean.head(10).to_string())

print("\n=== SUMMARY STATISTICS AFTER CLEANING (NUMERIC) ===")
print(df_clean[["age", "purchase_amount", "feedback_score"]].describe().round(2).to_string())

print("\n=== STANDARDISED CATEGORICAL COUNTS ===")
for col in ["gender", "city", "country"]:
    print(f"\n--- {col} ---")
    print(df_clean[col].value_counts(dropna=False).to_string())

print("\n=== CONTACT & TEMPORAL QUALITY SUMMARY ===")
print(f"Valid emails: {df_clean['has_valid_email'].sum():,} ({df_clean['has_valid_email'].mean()*100:.2f}%)")
print(f"Valid 10-digit phones: {df_clean['has_valid_phone'].sum():,} ({df_clean['has_valid_phone'].mean()*100:.2f}%)")
print(f"Both dates present: {(df_clean['signup_date'].notna() & df_clean['last_purchase_date'].notna()).sum():,}")
print(f"Inverted dates (last_purchase_date < signup_date): {df_clean['is_date_inverted'].sum():,}")


=== CLEANED DATASET HEAD(10) ===
  customer_id    name   gender   age       city signup_date last_purchase_date  purchase_amount  feedback_score           email phone_number    country  has_valid_email  has_valid_phone  is_date_inverted
0      C00001   Ankit   Female  36.0     Mumbai  2024-12-31         2025-08-13              NaN             2.0             NaN          NaN     Canada            False            False             False
1      C00002    Ravi   Female   NaN    Kolkata         NaT                NaT              NaN             NaN             NaN          NaN     Canada            False            False             False
2      C00003    Ravi   Female  66.0  Ahmedabad         NaT         2023-06-20              NaN            10.0  user2@mail.com          NaN      India             True            False             False
3      C00004   Ankit     Male  44.0    Kolkata         NaT         2025-09-13              NaN             NaN             NaN   9316267914        USA